In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score, classification_report
import io
from google.colab import files

In [ ]:
uploaded = files.upload()

Saving train_preprocessed.csv to train_preprocessed.csv


In [ ]:
train_df = pd.read_csv(io.BytesIO(uploaded['train_preprocessed.csv']))

In [ ]:
uploadd = files.upload()

Saving test_preprocessed.csv to test_preprocessed.csv


In [ ]:
test_df = pd.read_csv(io.BytesIO(uploadd['test_preprocessed.csv']))

In [ ]:
target_col = 'spend_category'
id_col = 'trip_id'

# Drop rows in training data where the target is missing
train_df = train_df.dropna(subset=[target_col])

# Separate target and features
X = train_df.drop([target_col, id_col], axis=1)
y = train_df[target_col]

test_X = test_df.drop([id_col, target_col], axis=1, errors='ignore')
test_X = test_X.reindex(columns=X.columns, fill_value=0)

# Scale Features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
test_X_scaled = scaler.transform(test_X)

# Encode Target
y = y.astype(int)
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

# Convert to One-Hot Encoding
num_classes = len(np.unique(y_encoded))
y_categorical = to_categorical(y_encoded, num_classes=num_classes)

# Split into Train and Validation
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_categorical, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Input Features: {X_train.shape[1]}")
print(f"Target Classes: {num_classes}")

Input Features: 200
Target Classes: 3


In [ ]:
# Define Neural Network Architecture
model = models.Sequential([
    # Input Layer
    layers.Input(shape=(X_train.shape[1],)),

    # Hidden Layer 1: Larger layer to capture patterns
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    # Hidden Layer 2
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    # Hidden Layer 3
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),

    # Output Layer: Softmax activation is standard for multi-class classification
    layers.Dense(num_classes, activation='softmax')
])

In [ ]:
# Compile Model
optimizer = optimizers.Adam(learning_rate=0.001)

model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Train Model
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.00001,
    verbose=1
)

print("\nStarting Training...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)


Starting Training...
Epoch 1/100
158/158 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.5752 - loss: 1.0164 - val_accuracy: 0.7417 - val_loss: 0.6561 - learning_rate: 0.0010
Epoch 2/100
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7138 - loss: 0.6877 - val_accuracy: 0.7445 - val_loss: 0.6160 - learning_rate: 0.0010
Epoch 3/100
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7401 - loss: 0.6202 - val_accuracy: 0.7484 - val_loss: 0.6077 - learning_rate: 0.0010
Epoch 4/100
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7466 - loss: 0.6016 - val_accuracy: 0.7492 - val_loss: 0.6037 - learning_rate: 0.0010
Epoch 5/100
158/158 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.7462 - loss: 0.5928 - val_accuracy: 0.7544 - val_loss: 0.6058 - learning_rate: 0.0010
Epoch 6/100
158/158 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.7578 - loss: 0.5844 - val_accuracy: 0.7500 - val_loss: 0.6032 - learning_rate: 0.0010
Epoch 7/100
158/158 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - ac

In [ ]:
# Evaluate
print("\nEvaluating on Validation Set...")
val_probs = model.predict(X_val)
val_preds = np.argmax(val_probs, axis=1)
val_true = np.argmax(y_val, axis=1)

f1 = f1_score(val_true, val_preds, average='macro')
print(f"\n>>> Validation Macro F1 Score: {f1:.4f} <<<")

print("\nClassification Report:")
print(classification_report(val_true, val_preds))


Evaluating on Validation Set...
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

>>> Validation Macro F1 Score: 0.6705 <<<

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.84      0.84      1249
           1       0.67      0.75      0.71       982
           2       0.68      0.36      0.47       293

    accuracy                           0.75      2524
   macro avg       0.73      0.65      0.67      2524
weighted avg       0.75      0.75      0.74      2524



In [ ]:
# Generate Submission
print("Generating predictions for submission...")
test_probs = model.predict(test_X_scaled)
test_preds_indices = np.argmax(test_probs, axis=1)

# Convert numeric predictions back to original labels
test_preds_labels = encoder.inverse_transform(test_preds_indices)

submission = pd.DataFrame({
    'trip_id': test_df['trip_id'],
    'spend_category': test_preds_labels
})

submission.to_csv('submission_nn.csv', index=False)
print("submission_nn.csv saved successfully!")

Generating predictions for submission...
183/183 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
submission_nn.csv saved successfully!
